# Integración de agricultura, clima y edafología (2003–2025)

Esta notebook construye de forma reproducible el conjunto municipal para análisis y modelado. Utiliza los archivos agrícolas oficiales normalizados por el paquete, clima mensual de NASA POWER y polígonos edafológicos de INEGI.

La unidad agrícola original se conserva: la integración no debe crear ni eliminar registros.

In [1]:
import geopandas as gpd
import pandas as pd

from proyecto_agronomia.config import (
    AGRICULTURA_RAW_DIR,
    ANIOS,
    ANIOS_MAPBIOMAS,
    EDAFOLOGIA_RAW_DIR,
    INEGI_INTERIM_DIR,
    MAPBIOMAS_INTERIM_DIR,
    MAPBIOMAS_RAW_DIR,
    NASA_POWER_PROCESSED_DIR,
    PROCESSED_DATA_DIR,
)
from datasets.mapbiomas import (
    CLAVE_NACIONAL,
    ruta_raster_mapbiomas,
)
from proyecto_agronomia.features import construir_historico_agricola #carga y normaliza los datos de agricultura
from processing.integracion_climatica import (
    construir_clima_mensual_ancho,
    crear_clave_municipal,
    integrar_agricultura_clima,
    resumir_clima_por_ciclo,
    validar_clima_mensual,
)
from processing.integracion_edafologica import (
    LLAVES_MASCARA,
    resumir_edafologia_municipal,
    resumir_edafologia_por_mascara,
)
from processing.integracion_mapbiomas import (
    asignar_mascara_mapbiomas,
    contar_suelo_por_mascara,
)
from processing.features_historicas import (
    COLUMNAS_HISTORICAS,
    agregar_historial_rendimiento,
)

RUTA_CLIMA_MENSUAL = (
    NASA_POWER_PROCESSED_DIR / "clima_municipal_mensual_2002_2025.csv"
)
RUTA_MUNICIPIOS = INEGI_INTERIM_DIR / "municipios_mexico_inegi.geojson"
RUTA_EDAFOLOGIA = (
    EDAFOLOGIA_RAW_DIR / "serie_iii" / "conjunto_de_datos"
    / "conj_nac_inf_edaf_esc_250k_ser_III_area.shp"
)
RUTA_EDAFOLOGIA_MUNICIPAL = PROCESSED_DATA_DIR / "edafologia_municipal.csv"
RUTA_EDAFOLOGIA_MASCARA = PROCESSED_DATA_DIR / "edafologia_mascara_mapbiomas.csv"
RUTA_AGRICULTURA_CLIMA = PROCESSED_DATA_DIR / "agricultura_clima_2003_2025.csv"
RUTA_FINAL = (
    PROCESSED_DATA_DIR / "agricultura_clima_edafologia_2003_2025.csv"
)
RUTA_FINAL_HISTORIAL = (
    PROCESSED_DATA_DIR / "agricultura_clima_edafologia_historial_2003_2025.csv"
)
RECALCULAR_EDAFOLOGIA_MUNICIPAL = False
RECALCULAR_EDAFOLOGIA_MASCARA = False
# Área mínima (ha) de la máscara MapBiomas para usar su composición de suelo;
# por debajo se usa la del municipio completo.
UMBRAL_MASCARA_HA = 1.0
pd.set_option("display.max_columns", None)

2026-11-24 16:14:43.001 | INFO     | proyecto_agronomia.config:<module>:14 - PROJ_ROOT path is: B:\MCD2\JW\agro\proyecto_agronomia


## Cargar y normalizar la producción agrícola

Los archivos `agricola_*.csv` se leen desde `data/raw/agricultura`. La normalización resuelve cambios históricos de columnas, nombres de cultivo, precios, separadores de miles y errores numéricos.

In [2]:
df_agricultura = construir_historico_agricola(
    anios=ANIOS,
    directorio=AGRICULTURA_RAW_DIR,
)
df_agricultura = df_agricultura.drop(
    columns=[c for c in df_agricultura if c.startswith("Unnamed:")],
    errors="ignore",
)
df_agricultura = crear_clave_municipal(df_agricultura)
df_agricultura["Nomcicloproductivo"] = (
    df_agricultura["Nomcicloproductivo"].astype("string").str.strip()
)

columnas_agricolas_requeridas = {
    "Anio", "cvegeo", "Nomestado", "Nommunicipio",
    "Nomcultivo", "Nomcicloproductivo", "Nommodalidad",
    "Sembrada", "Cosechada", "Siniestrada",
    "Volumenproduccion", "Rendimiento",
}
faltantes = columnas_agricolas_requeridas - set(df_agricultura.columns)
assert not faltantes, f"Faltan columnas agrícolas: {sorted(faltantes)}"
assert set(df_agricultura["Anio"].unique()) == set(ANIOS)
assert df_agricultura["cvegeo"].str.fullmatch(r"\d{5}").all()

ciclos_validos = {"Otoño-Invierno", "Primavera-Verano", "Perennes"}
ciclos_encontrados = set(df_agricultura["Nomcicloproductivo"].dropna())
assert ciclos_encontrados <= ciclos_validos, (
    f"Ciclos no contemplados: {sorted(ciclos_encontrados-ciclos_validos)}"
)

print(f"Registros agrícolas: {len(df_agricultura):,}")
display(
    df_agricultura.groupby(["Anio", "Nomcicloproductivo"])
    .agg(
        registros=("cvegeo", "size"),
        municipios=("cvegeo", "nunique"),
        cultivos=("Nomcultivo", "nunique"),
    )
)

Procesando datos agrícolas:   0%|          | 0/23 [00:00<?, ?it/s]

2026-11-24 16:14:47.827 | DEBUG    | proyecto_agronomia.features:normalizar_columnas_agricolas:84 - [2003] Columnas normalizadas.


Procesando datos agrícolas:   4%|▍         | 1/23 [00:00<00:16,  1.35it/s]

2026-11-24 16:14:48.204 | INFO     | proyecto_agronomia.features:construir_historico_agricola:180 - [2003] 35,667 registros procesados.


Procesando datos agrícolas:   4%|▍         | 1/23 [00:01<00:16,  1.35it/s]

2026-11-24 16:14:48.534 | DEBUG    | proyecto_agronomia.features:normalizar_columnas_agricolas:84 - [2004] Columnas normalizadas.


... (83 outputs left out)

2026-11-24 16:15:03.398 | DEBUG    | proyecto_agronomia.features:normalizar_columnas_agricolas:84 - [2025] Columnas normalizadas.


Procesando datos agrícolas: 100%|██████████| 23/23 [00:16<00:00,  1.41it/s]


2026-11-24 16:15:03.782 | INFO     | proyecto_agronomia.features:construir_historico_agricola:180 - [2025] 35,902 registros procesados.
Registros agrícolas: 825,089


registros  municipios  cultivos
Anio Nomcicloproductivo                                 
2003 Otoño-Invierno           9858        1897       147
     Perennes                10488        2118       128
     Primavera-Verano        15321        2362       159
2004 Otoño-Invierno           9935        1902       155
     Perennes                10838        2130       129
...                            ...         ...       ...
2024 Perennes                11631        2290       132
     Primavera-Verano        16077        2406       162
2025 Otoño-Invierno           9099        1923       146
     Perennes                11763        2292       132
     Primavera-Verano        15040        2393       156

[69 rows x 3 columns]

## Cargar y validar el clima mensual NASA POWER

La llave climática es `cvegeo + anio + mes`. El año 2002 se necesita para formar octubre–diciembre del ciclo Otoño–Invierno de 2003.

In [3]:
if not RUTA_CLIMA_MENSUAL.exists():
    raise FileNotFoundError(
        f"No existe {RUTA_CLIMA_MENSUAL}. Ejecuta descarga_clima_nasa.ipynb."
    )

columnas_clima = [
    "cvegeo", "anio", "mes", "temperatura_media_mes_c",
    "precipitacion_mes_mm", "dias_esperados",
    "dias_temperatura", "dias_precipitacion",
]
df_clima_mensual = pd.read_csv(
    RUTA_CLIMA_MENSUAL,
    usecols=columnas_clima,
    dtype={"cvegeo": "string"},
)
df_clima_mensual = validar_clima_mensual(df_clima_mensual)
print(f"Registros climáticos mensuales: {len(df_clima_mensual):,}")
display(
    df_clima_mensual.groupby("anio").agg(
        municipios=("cvegeo", "nunique"),
        meses=("mes", "count"),
        temperatura_nulos=("temperatura_media_mes_c", lambda s: s.isna().sum()),
        precipitacion_nulos=("precipitacion_mes_mm", lambda s: s.isna().sum()),
    )
)

Registros climáticos mensuales: 713,664


,municipios,meses,temperatura_nulos,precipitacion_nulos
anio,,,,
2002,2478,29736,0,0
2003,2478,29736,0,0
2004,2478,29736,0,0
2005,2478,29736,0,0
2006,2478,29736,0,0
2007,2478,29736,0,0
2008,2478,29736,0,0
2009,2478,29736,0,0
2010,2478,29736,0,0


## Construir características climáticas

Se conservan los meses como columnas y se calculan ventanas compatibles con los ciclos productivos:

- Otoño–Invierno: octubre–diciembre anterior y enero–marzo actual.
- Primavera–Verano: abril–septiembre.
- Perennes: enero–diciembre.

In [4]:
df_clima_ancho = construir_clima_mensual_ancho(df_clima_mensual)
df_clima_ciclos = resumir_clima_por_ciclo(df_clima_mensual)
df_clima_ciclos = df_clima_ciclos.loc[df_clima_ciclos["anio"].isin(ANIOS)]

assert not df_clima_ancho.duplicated(["cvegeo", "anio"]).any()
assert not df_clima_ciclos.duplicated(
    ["cvegeo", "anio", "Nomcicloproductivo"]
).any()
assert df_clima_ciclos[[
    "temperatura_media_ciclo_c", "precipitacion_ciclo_mm"
]].notna().all().all()

display(
    df_clima_ciclos.groupby(["anio", "Nomcicloproductivo"])
    .agg(
        municipios=("cvegeo", "nunique"),
        dias_min=("dias_clima_ciclo", "min"),
        dias_max=("dias_clima_ciclo", "max"),
    )
)

municipios  dias_min  dias_max
anio Nomcicloproductivo                                
2003 Otoño-Invierno            2478       182       182
     Perennes                  2478       365       365
     Primavera-Verano          2478       183       183
2004 Otoño-Invierno            2478       183       183
     Perennes                  2478       366       366
...                             ...       ...       ...
2024 Perennes                  2478       366       366
     Primavera-Verano          2478       183       183
2025 Otoño-Invierno            2478       182       182
     Perennes                  2478       365       365
     Primavera-Verano          2478       183       183

[69 rows x 3 columns]

## Integrar agricultura y clima

Las uniones son `many_to_one`; por lo tanto, el número de registros agrícolas debe permanecer sin cambios y toda observación debe recibir clima.

In [5]:
df_agricultura_clima = integrar_agricultura_clima(
    df_agricultura,
    df_clima_ciclos,
    df_clima_ancho,
)
columnas_climaticas_finales = [
    columna for columna in df_agricultura_clima.columns
    if columna.startswith(("temperatura_", "precipitacion_"))
]
assert len(df_agricultura_clima) == len(df_agricultura)
assert df_agricultura_clima[columnas_climaticas_finales].notna().all().all()

resumen_cobertura = (
    df_agricultura_clima.groupby(["Anio", "Nomcicloproductivo"])
    .agg(
        registros=("cvegeo", "size"),
        municipios=("cvegeo", "nunique"),
        temperatura=("temperatura_media_ciclo_c", "count"),
        precipitacion=("precipitacion_ciclo_mm", "count"),
    )
)
resumen_cobertura["cobertura_pct"] = (
    100 * resumen_cobertura["temperatura"] / resumen_cobertura["registros"]
).round(2)
assert resumen_cobertura["cobertura_pct"].eq(100).all()
display(resumen_cobertura)

RUTA_AGRICULTURA_CLIMA.parent.mkdir(parents=True, exist_ok=True)
df_agricultura_clima.to_csv(
    RUTA_AGRICULTURA_CLIMA, index=False, encoding="utf-8"
)
print(f"Dataset agricultura-clima: {RUTA_AGRICULTURA_CLIMA}")
print(f"Dimensiones: {df_agricultura_clima.shape}")

registros  municipios  temperatura  precipitacion  \
Anio Nomcicloproductivo                                                      
2003 Otoño-Invierno           9858        1897         9858           9858   
     Perennes                10488        2118        10488          10488   
     Primavera-Verano        15321        2362        15321          15321   
2004 Otoño-Invierno           9935        1902         9935           9935   
     Perennes                10838        2130        10838          10838   
...                            ...         ...          ...            ...   
2024 Perennes                11631        2290        11631          11631   
     Primavera-Verano        16077        2406        16077          16077   
2025 Otoño-Invierno           9099        1923         9099           9099   
     Perennes                11763        2292        11763          11763   
     Primavera-Verano        15040        2393        15040          15040   

                         cobertura_pct  
Anio Nomcicloproductivo                 
2003 Otoño-Invierno              100.0  
     Perennes                    100.0  
     Primavera-Verano            100.0  
2004 Otoño-Invierno              100.0  
     Perennes                    100.0  
...                                ...  
2024 Perennes                    100.0  
     Primavera-Verano            100.0  
2025 Otoño-Invierno              100.0  
     Perennes                    100.0  
     Primavera-Verano            100.0  

[69 rows x 5 columns]

Dataset agricultura-clima: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\agricultura_clima_2003_2025.csv
Dimensiones: (825089, 58)


## Construir o cargar el resumen edafológico municipal

La intersección espacial puede tardar. Una vez generada, `edafologia_municipal.csv` se reutiliza. Cambia `RECALCULAR_EDAFOLOGIA_MUNICIPAL` a `True` solamente para reconstruirla. Si la caché es de una versión anterior (sin columnas `suelo_pct_*` o con columnas de fases), se reconstruye automáticamente.

Además del grupo de suelo dominante y la textura dominante, el resumen incluye la **composición edafológica**: una columna `suelo_pct_<código>` por cada grupo de suelo WRB con el porcentaje del área municipal que ocupa (0 si no está presente). `SA` (suelo cubierto por agua) se trata como un grupo más. Las fases física y química no se incluyen.

Este resumen municipal ya no es la fuente principal del suelo: se usa como **respaldo** del filtro MapBiomas de la siguiente sección.

In [6]:
def _cache_edafologia_vigente(ruta):
    """La caché es válida si ya trae la composición y no trae fases."""
    if not ruta.exists():
        return False
    columnas = pd.read_csv(ruta, nrows=0).columns
    tiene_composicion = any(c.startswith("suelo_pct_") for c in columnas)
    tiene_fases = any(c.startswith("fase_") for c in columnas)
    return tiene_composicion and not tiene_fases


if _cache_edafologia_vigente(RUTA_EDAFOLOGIA_MUNICIPAL) and not RECALCULAR_EDAFOLOGIA_MUNICIPAL:
    df_edafologia_municipal = pd.read_csv(
        RUTA_EDAFOLOGIA_MUNICIPAL, dtype={"cvegeo": "string"}
    )
    df_edafologia_municipal["cvegeo"] = (
        df_edafologia_municipal["cvegeo"].str.zfill(5)
    )
else:
    for ruta in [RUTA_MUNICIPIOS, RUTA_EDAFOLOGIA]:
        if not ruta.exists():
            raise FileNotFoundError(f"No se encontró el archivo requerido: {ruta}")
    municipios = gpd.read_file(RUTA_MUNICIPIOS)
    edafologia = gpd.read_file(RUTA_EDAFOLOGIA)
    df_edafologia_municipal = resumir_edafologia_municipal(
        municipios, edafologia
    )
    df_edafologia_municipal.to_csv(
        RUTA_EDAFOLOGIA_MUNICIPAL, index=False, encoding="utf-8"
    )

COLUMNAS_COMPOSICION_SUELO = [
    c for c in df_edafologia_municipal.columns if c.startswith("suelo_pct_")
]
assert not df_edafologia_municipal["cvegeo"].duplicated().any()
print(f"Municipios edafológicos: {len(df_edafologia_municipal):,}")
print(f"Columnas de composición de suelo: {len(COLUMNAS_COMPOSICION_SUELO)}")
display(df_edafologia_municipal.head())

Municipios edafológicos: 2,478
Columnas de composición de suelo: 29


,cvegeo,grupo_suelo_dominante,grupo_suelo_dominante_pct,textura_dominante,textura_dominante_pct,grupos_suelo_n,suelo_pct_AC,suelo_pct_AL,suelo_pct_AN,suelo_pct_AR,suelo_pct_AT,suelo_pct_CH,suelo_pct_CL,suelo_pct_CM,suelo_pct_DU,suelo_pct_FL,suelo_pct_GL,suelo_pct_GY,suelo_pct_HS,suelo_pct_KS,suelo_pct_LP,suelo_pct_LV,suelo_pct_LX,suelo_pct_NT,suelo_pct_PH,suelo_pct_PL,suelo_pct_PT,suelo_pct_RG,suelo_pct_SA,suelo_pct_SC,suelo_pct_SN,suelo_pct_ST,suelo_pct_TC,suelo_pct_UM,suelo_pct_VR,nombre_suelo_wrb
0,01001,PH,24.72,2.0,88.81,11,2.01,0.0,0.0,0.0,0.0,0.0,10.27,11.59,5.24,1.19,0.0,0.0,0.0,1.85,15.39,0.00,0.0,0.0,24.72,16.55,0.0,0.00,0.37,0.0,0.0,0.0,10.81,0.0,0.0,Phaeozem
1,01008,LP,31.44,2.0,95.48,10,4.85,0.0,0.0,0.0,0.0,0.0,0.00,18.40,0.00,0.00,0.0,0.0,0.0,9.67,31.44,11.21,0.0,0.0,16.01,4.95,0.0,0.49,2.65,0.0,0.0,0.0,0.32,0.0,0.0,Leptosol
2,01011,DU,41.24,2.0,93.67,7,0.00,0.0,0.0,0.0,0.0,0.0,2.84,0.00,41.24,3.92,0.0,0.0,0.0,20.16,8.14,0.00,0.0,0.0,17.38,0.00,0.0,0.00,0.00,0.0,0.0,0.0,6.33,0.0,0.0,Durisol
3,01009,DU,38.81,2.0,77.62,6,0.00,0.0,0.0,0.0,0.0,0.0,2.65,0.00,38.81,3.67,0.0,0.0,0.0,0.00,31.34,0.00,0.0,0.0,0.00,0.00,0.0,21.03,0.00,0.0,0.0,0.0,2.50,0.0,0.0,Durisol
4,01007,DU,39.54,2.0,98.41,11,0.00,0.0,0.0,0.0,0.0,0.0,0.00,12.72,39.54,2.65,0.0,0.0,0.0,1.29,16.91,1.91,0.0,0.0,13.90,0.72,0.0,8.82,0.30,0.0,0.0,0.0,1.24,0.0,0.0,Durisol


## Filtrar el suelo por la cobertura agrícola de MapBiomas

El municipio completo incluye bosques, matorrales, zonas urbanas, etc. Para aproximar mejor el suelo donde se siembra, se usa la cobertura anual de **MapBiomas México (Colección 1, 30 m, CC BY 4.0)**. Cada registro agrícola se asigna a una máscara según su tipo de cultivo:

| Máscara | Clase MapBiomas | Registros SIAP |
|---|---|---|
| `anual` | 19 Cultivo anual | ciclos Primavera–Verano y Otoño–Invierno |
| `perenne` | 36 Cultivo perenne | ciclo Perennes |
| `pastizal` | 15 Pastizal cultivado e inducido | pastos y zacates (`Pastos y praderas`, `Pasto tapete`, semillas de pasto/zacate), sin importar el ciclo |

Todos los cultivos del mismo tipo, municipio y año comparten los mismos datos edafológicos. Como el clima, esto es una **aproximación**: no se exige que el área de la máscara coincida con la superficie sembrada de SIAP. MapBiomas mide la tierra agrícola, incluido el barbecho, y SIAP lo sembrado; además, una parcela puede tener varios cultivos en el año.

- Un píxel pertenece a la máscara del año *t* si tuvo la clase en *t−1*, *t* o *t+1*. Así se absorbe el ruido interanual de la clasificación y los barbechos. Por eso se descarga desde 2002; en 2025 la ventana queda en 2024–2025.
- La composición de suelo (`suelo_pct_*`, dominantes, textura) se calcula como % del área de la máscara en el municipio y el año.
- **Respaldos** cuando la máscara propia mide menos de `UMBRAL_MASCARA_HA`:
  - perennes → máscara `agricola` (clases 19+36), porque MapBiomas clasifica como cultivo anual buena parte de la caña, los cítricos, la alfalfa y la vid;
  - si tampoco hay, o en anuales y pastizales, el resumen municipal.
- `suelo_fuente` indica qué se usó (`mascara`, `agricola` o `municipal`) y `area_mascara_ha` el tamaño de la máscara propia.
- La clase 21 *Mosaico de usos* no se usa: prácticamente no aparece en la Colección 1.

Los rasters se descargan con `descarga_mapbiomas.ipynb` en `data/raw/mapbiomas/00/<año>.tif`. El cruce raster × municipios × suelos se guarda por estado en `data/interim/mapbiomas/suelo_por_mascara/` y se reanuda si se interrumpe (unos 15 minutos la primera vez).

In [7]:
df_agricultura_clima["mascara"] = asignar_mascara_mapbiomas(
    df_agricultura_clima["Nomcultivo"], df_agricultura_clima["Nomcicloproductivo"]
)
assert df_agricultura_clima["mascara"].notna().all()
display(
    df_agricultura_clima.groupby(["mascara", "Nomcicloproductivo"], observed=True)
    .agg(registros=("cvegeo", "size"), sembrada_ha=("Sembrada", "sum"))
    .round(0)
)

llaves_agricolas = df_agricultura_clima[LLAVES_MASCARA].drop_duplicates()


RUTA_PARAMETROS_CRUCE = MAPBIOMAS_INTERIM_DIR / "suelo_por_mascara" / "parametros.json"


def _cache_mascara_vigente(ruta):
    """Vigente si es más nueva que el cruce raster (cuyos parámetros cambian al
    cambiar máscaras, años o ventana) y trae las columnas actuales."""
    if not (ruta.exists() and RUTA_PARAMETROS_CRUCE.exists()):
        return False
    if ruta.stat().st_mtime < RUTA_PARAMETROS_CRUCE.stat().st_mtime:
        return False
    columnas = set(pd.read_csv(ruta, nrows=0).columns)
    return {"mascara", "suelo_fuente", "area_mascara_ha"} <= columnas


if _cache_mascara_vigente(RUTA_EDAFOLOGIA_MASCARA) and not RECALCULAR_EDAFOLOGIA_MASCARA:
    df_edafologia = pd.read_csv(
        RUTA_EDAFOLOGIA_MASCARA, dtype={"cvegeo": "string", "mascara": "string"}
    )
    df_edafologia["cvegeo"] = df_edafologia["cvegeo"].str.zfill(5)
else:
    faltan_rasters = [
        anio for anio in ANIOS_MAPBIOMAS
        if not ruta_raster_mapbiomas(CLAVE_NACIONAL, anio, MAPBIOMAS_RAW_DIR).exists()
    ]
    if faltan_rasters:
        raise FileNotFoundError(
            f"Faltan los rasters MapBiomas de {faltan_rasters}. "
            "Ejecuta primero notebooks/descarga_mapbiomas.ipynb."
        )
    municipios = gpd.read_file(RUTA_MUNICIPIOS)
    edafologia = gpd.read_file(RUTA_EDAFOLOGIA, columns=["Grupo1", "Clase_tex"])
    conteos_mascara = contar_suelo_por_mascara(
        municipios, edafologia, anios=ANIOS, recalcular=RECALCULAR_EDAFOLOGIA_MASCARA
    )
    df_edafologia = resumir_edafologia_por_mascara(
        conteos_mascara,
        df_edafologia_municipal,
        llaves_agricolas,
        umbral_ha=UMBRAL_MASCARA_HA,
    )
    df_edafologia.to_csv(RUTA_EDAFOLOGIA_MASCARA, index=False, encoding="utf-8")

COLUMNAS_COMPOSICION_SUELO = [
    c for c in df_edafologia.columns if c.startswith("suelo_pct_")
]
assert not df_edafologia.duplicated(LLAVES_MASCARA).any()
faltan_llaves = llaves_agricolas.merge(
    df_edafologia[LLAVES_MASCARA], how="left", indicator=True
)["_merge"].eq("left_only")
assert not faltan_llaves.any(), f"{faltan_llaves.sum()} llaves agrícolas sin resumen."

print(f"Combinaciones municipio-año-máscara: {len(df_edafologia):,}")
print("Fuente del suelo (% de combinaciones):")
display(
    pd.crosstab(df_edafologia["mascara"], df_edafologia["suelo_fuente"], normalize="index")
    .mul(100).round(1)
)
display(
    df_edafologia.groupby("mascara")["area_mascara_ha"]
    .describe(percentiles=[0.1, 0.5, 0.9]).round(1)
)

registros  sembrada_ha
mascara  Nomcicloproductivo                        
anual    Otoño-Invierno         212735   84327376.0
         Primavera-Verano       353974  259363067.0
pastizal Otoño-Invierno           3735     505070.0
         Perennes                25627   50593394.0
         Primavera-Verano          867     342682.0
perenne  Perennes               228151   91550422.0

2026-11-24 16:16:31.356 | INFO     | processing.integracion_mapbiomas:contar_suelo_por_mascara:304 - MapBiomas × suelo: estado 01 (11 municipios).
2026-11-24 16:16:45.534 | INFO     | processing.integracion_mapbiomas:contar_suelo_por_mascara:304 - MapBiomas × suelo: estado 02 (7 municipios).
2026-11-24 16:18:43.151 | INFO     | processing.integracion_mapbiomas:contar_suelo_por_mascara:304 - MapBiomas × suelo: estado 03 (5 municipios).
2026-11-24 16:20:58.703 | INFO     | processing.integracion_mapbiomas:contar_suelo_por_mascara:304 - MapBiomas × suelo: estado 04 (13 municipios).
2026-11-24 16:22:18.340 | INFO     | processing.integracion_mapbiomas:contar_suelo_por_mascara:304 - MapBiomas × suelo: estado 05 (38 municipios).
2026-11-24 16:24:51.703 | INFO     | processing.integracion_mapbiomas:contar_suelo_por_mascara:304 - MapBiomas × suelo: estado 06 (10 municipios).
2026-11-24 16:25:09.837 | INFO     | processing.integracion_mapbiomas:contar_suelo_por_mascara:304 - MapBiomas × suelo: 

suelo_fuente,mascara,agricola,municipal
mascara,,,
anual,98.8,0.0,1.2
pastizal,95.5,0.0,4.5
perenne,57.3,42.1,0.6


,count,mean,std,min,10%,50%,90%,max
mascara,,,,,,,,
anual,55718.0,15581.1,42056.4,0.0,102.3,3790.0,36563.1,980470.8
pastizal,20133.0,6806.8,17020.9,0.0,15.4,1168.6,16913.4,269974.8
perenne,49728.0,247.2,1196.3,0.0,0.0,2.7,350.8,30806.0


## Integrar suelo y guardar el dataset final

El resumen edafológico tiene una fila por municipio, año y máscara (`cvegeo`, `Anio`, `mascara`), así que la unión tampoco debe multiplicar registros agrícolas.

In [8]:
df_final = df_agricultura_clima.merge(
    df_edafologia,
    on=LLAVES_MASCARA,
    how="left",
    validate="many_to_one",
)
assert len(df_final) == len(df_agricultura), (
    "La integración edafológica alteró el número de registros."
)
columnas_suelo = [
    "grupo_suelo_dominante", "nombre_suelo_wrb",
    "grupo_suelo_dominante_pct", "grupos_suelo_n",
    "textura_dominante", "area_mascara_ha", "suelo_fuente",
]
faltantes_suelo = df_final["grupo_suelo_dominante"].isna()
if faltantes_suelo.any():
    claves_faltantes = sorted(df_final.loc[faltantes_suelo, "cvegeo"].unique())
    raise ValueError(
        f"Hay {len(claves_faltantes)} municipios agrícolas sin suelo: "
        f"{claves_faltantes[:20]}"
    )

# La composición se expresa sobre el área de la máscara (o del municipio en el
# respaldo), por lo que no puede superar 100 %. Puede quedar por debajo donde
# la capa edafológica no cubre lagunas o mar.
assert df_final[COLUMNAS_COMPOSICION_SUELO].notna().all().all()
suma_composicion = df_edafologia[COLUMNAS_COMPOSICION_SUELO].sum(axis=1)
assert suma_composicion.le(100.5).all(), suma_composicion.describe()
print("Cobertura edafológica (% del área de referencia):")
display(
    suma_composicion.groupby(df_edafologia["suelo_fuente"]).describe().round(2)
)
print(f"Combinaciones con cobertura < 95 %: {suma_composicion.lt(95).sum():,}")

print("Registros agrícolas por fuente de suelo (%):")
display(
    pd.crosstab(df_final["mascara"], df_final["suelo_fuente"], normalize="index")
    .mul(100).round(1)
)

df_final.to_csv(RUTA_FINAL, index=False, encoding="utf-8")
print(f"Dataset final: {RUTA_FINAL}")
print(f"Registros: {len(df_final):,}")
print(f"Columnas: {len(df_final.columns):,}")
print(f"Periodo: {df_final['Anio'].min()}–{df_final['Anio'].max()}")
display(
    df_final[[
        "Anio", "cvegeo", "Nomestado", "Nommunicipio",
        "Nomcultivo", "Nomcicloproductivo", "mascara",
        "temperatura_media_ciclo_c", "precipitacion_ciclo_mm",
        *columnas_suelo, *COLUMNAS_COMPOSICION_SUELO,
    ]].head()
)

Cobertura edafológica (% del área de referencia):


,count,mean,std,min,25%,50%,75%,max
suelo_fuente,,,,,,,,
mascara,102761.0,99.92,1.45,7.03,100.00,100.0,100.0,100.03
agricola,20919.0,99.94,0.82,66.32,100.00,100.0,100.0,100.03
municipal,1899.0,99.72,2.36,61.92,99.99,100.0,100.0,100.03


Combinaciones con cobertura < 95 %: 317
Registros agrícolas por fuente de suelo (%):


suelo_fuente,mascara,agricola,municipal
mascara,,,
anual,99.3,0.0,0.7
pastizal,94.8,0.0,5.2
perenne,66.9,32.7,0.4


Dataset final: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\agricultura_clima_edafologia_2003_2025.csv
Registros: 825,089
Columnas: 96
Periodo: 2003–2025


,Anio,cvegeo,Nomestado,Nommunicipio,Nomcultivo,Nomcicloproductivo,mascara,temperatura_media_ciclo_c,precipitacion_ciclo_mm,grupo_suelo_dominante,nombre_suelo_wrb,grupo_suelo_dominante_pct,grupos_suelo_n,textura_dominante,area_mascara_ha,suelo_fuente,suelo_pct_AC,suelo_pct_AL,suelo_pct_AN,suelo_pct_AR,suelo_pct_AT,suelo_pct_CH,suelo_pct_CL,suelo_pct_CM,suelo_pct_DU,suelo_pct_FL,suelo_pct_GL,suelo_pct_GY,suelo_pct_HS,suelo_pct_KS,suelo_pct_LP,suelo_pct_LV,suelo_pct_LX,suelo_pct_NT,suelo_pct_PH,suelo_pct_PL,suelo_pct_PT,suelo_pct_RG,suelo_pct_SA,suelo_pct_SC,suelo_pct_SN,suelo_pct_ST,suelo_pct_TC,suelo_pct_UM,suelo_pct_VR
0,2003,01001,Aguascalientes,Aguascalientes,Avena forrajera en verde,Otoño-Invierno,anual,13.835165,84.8,PH,Phaeozem,27.10,11,2.0,75192.04,mascara,0.71,0.0,0.0,0.0,0.0,0.0,10.87,17.39,5.23,1.70,0.0,0.0,0.0,1.27,7.67,0.0,0.0,0.0,27.10,24.31,0.0,0.0,0.25,0.0,0.0,0.0,3.5,0.0,0.0
1,2003,01001,Aguascalientes,Aguascalientes,Cebada forrajera en verde,Otoño-Invierno,anual,13.835165,84.8,PH,Phaeozem,27.10,11,2.0,75192.04,mascara,0.71,0.0,0.0,0.0,0.0,0.0,10.87,17.39,5.23,1.70,0.0,0.0,0.0,1.27,7.67,0.0,0.0,0.0,27.10,24.31,0.0,0.0,0.25,0.0,0.0,0.0,3.5,0.0,0.0
2,2003,01001,Aguascalientes,Aguascalientes,Centeno forrajero en verde,Otoño-Invierno,anual,13.835165,84.8,PH,Phaeozem,27.10,11,2.0,75192.04,mascara,0.71,0.0,0.0,0.0,0.0,0.0,10.87,17.39,5.23,1.70,0.0,0.0,0.0,1.27,7.67,0.0,0.0,0.0,27.10,24.31,0.0,0.0,0.25,0.0,0.0,0.0,3.5,0.0,0.0
3,2003,01001,Aguascalientes,Aguascalientes,Triticale forrajero en verde,Otoño-Invierno,anual,13.835165,84.8,PH,Phaeozem,27.10,11,2.0,75192.04,mascara,0.71,0.0,0.0,0.0,0.0,0.0,10.87,17.39,5.23,1.70,0.0,0.0,0.0,1.27,7.67,0.0,0.0,0.0,27.10,24.31,0.0,0.0,0.25,0.0,0.0,0.0,3.5,0.0,0.0
4,2003,01001,Aguascalientes,Aguascalientes,Pastos y praderas,Otoño-Invierno,pastizal,13.835165,84.8,PH,Phaeozem,55.01,11,2.0,1287.30,mascara,1.55,0.0,0.0,0.0,0.0,0.0,29.20,0.03,3.65,0.17,0.0,0.0,0.0,0.05,8.47,0.0,0.0,0.0,55.01,1.77,0.0,0.0,0.01,0.0,0.0,0.0,0.1,0.0,0.0


In [9]:
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 825089 entries, 0 to 825088
Data columns (total 96 columns):
 #   Column                         Non-Null Count   Dtype   
---  ------                         --------------   -----   
 0   Anio                           825089 non-null  int64   
 1   Idestado                       825089 non-null  int64   
 2   Nomestado                      825089 non-null  str     
 3   Idddr                          825089 non-null  int64   
 4   Nomddr                         825089 non-null  str     
 5   Idcader                        825089 non-null  int64   
 6   Nomcader                       825089 non-null  str     
 7   Idmunicipio                    825089 non-null  int64   
 8   Nommunicipio                   825089 non-null  str     
 9   Idciclo                        825089 non-null  int64   
 10  Nomcicloproductivo             825089 non-null  object  
 11  Idmodalidad                    825089 non-null  int64   
 12  Nommodalidad               

## Agregar estabilidad histórica sin fuga temporal

Para cada municipio, cultivo, ciclo, modalidad y unidad, las características del año actual utilizan exclusivamente rendimientos de años anteriores. Los primeros años permanecen sin historial suficiente.

In [10]:
df_final_historial = agregar_historial_rendimiento(df_final, minimo_periodos=3)
assert len(df_final_historial) == len(df_final)
assert df_final_historial.loc[
    df_final_historial['Anio'].eq(df_final_historial['Anio'].min()),
    'rendimiento_hist_media',
].isna().all()
df_final_historial.to_csv(RUTA_FINAL_HISTORIAL, index=False, encoding='utf-8')
print(f'Dataset con historial: {RUTA_FINAL_HISTORIAL}')
display(
    df_final_historial[[
        'Anio', 'cvegeo', 'Nomcultivo', 'Nomcicloproductivo',
        'Nommodalidad', *COLUMNAS_HISTORICAS,
    ]].loc[lambda tabla: tabla['rendimiento_hist_media'].notna()].head()
)

Dataset con historial: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\agricultura_clima_edafologia_historial_2003_2025.csv


,Anio,cvegeo,Nomcultivo,Nomcicloproductivo,Nommodalidad,rendimiento_hist_anios,rendimiento_anio_anterior,rendimiento_hist_media,rendimiento_hist_desviacion,rendimiento_hist_cv
108391,2006,01001,Avena forrajera en verde,Otoño-Invierno,Riego,3,26.370840,26.702666,0.287415,0.010764
108393,2006,01001,Triticale forrajero en verde,Otoño-Invierno,Riego,3,25.333333,25.111111,5.003702,0.199262
108394,2006,01001,Pastos y praderas,Otoño-Invierno,Riego,3,59.918715,60.947866,1.183411,0.019417
108396,2006,01001,Chile verde,Primavera-Verano,Riego,3,6.485714,6.281501,0.348771,0.055524
108397,2006,01001,Coliflor,Primavera-Verano,Riego,3,12.000000,11.291667,1.120361,0.099220


## Recargar y validar el producto

Esta última lectura verifica que el CSV persistido conserva la llave municipal, el periodo y las variables esenciales.

In [11]:
df_verificacion = pd.read_csv(
    RUTA_FINAL_HISTORIAL, dtype={"cvegeo": "string"}, low_memory=False
)
df_verificacion["cvegeo"] = df_verificacion["cvegeo"].str.zfill(5)
assert len(df_verificacion) == len(df_final)
assert set(df_verificacion["Anio"].unique()) == set(ANIOS)
assert df_verificacion["cvegeo"].str.fullmatch(r"\d{5}").all()
assert df_verificacion[[
    "temperatura_media_ciclo_c", "precipitacion_ciclo_mm",
    "grupo_suelo_dominante",
]].notna().all().all()
assert set(COLUMNAS_HISTORICAS).issubset(df_verificacion.columns)
display(pd.DataFrame({
    "archivo": [str(RUTA_FINAL_HISTORIAL)],
    "registros": [len(df_verificacion)],
    "columnas": [len(df_verificacion.columns)],
    "municipios": [df_verificacion["cvegeo"].nunique()],
    "cultivos": [df_verificacion["Nomcultivo"].nunique()],
    "anio_inicial": [df_verificacion["Anio"].min()],
    "anio_final": [df_verificacion["Anio"].max()],
}))

,archivo,registros,columnas,municipios,cultivos,anio_inicial,anio_final
0,B:\MCD2\JW\agro\proyecto_agronomia\data\proces...,825089,101,2452,385,2003,2025
